In [1]:
import numpy as np
from qutip import *
import plotly.graph_objects as go
from tqdm import tqdm,trange
import os
from datetime import datetime
timestamp = datetime.now().strftime('%M-%S')

from scipy.interpolate import interp1d
from sklearn.metrics import mean_absolute_error

In [2]:
# --- Updated signal function ---
def signal(data, samples, first='reference'):
    data_shape = data.shape[0]
    steps = int(data_shape / (2 * samples))

    if first.lower() == 'reference':
        reference_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        signal_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    elif first.lower() == 'signal':
        signal_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        reference_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    else:
        raise ValueError("`first` must be either 'reference' or 'signal'")

    signal_photon = signal_samples / reference_samples
    return signal_photon, reference_samples, signal_samples

# --- Wrapper function ---
def data_to_time_signal(data, samples, first='reference'):
    time = data['time_axis']
    signal_photon, reference_samples, signal_samples = signal(data['avg_data'], samples, first=first)
    return time, signal_photon, reference_samples, signal_samples

In [3]:
from plotly import graph_objs as go
# Figure format for plotly
fig_template = go.layout.Template()
fig_template.layout = {
    'template': 'simple_white+presentation',
    'autosize': False,
    'width': 1120,
    'height': 450,
    # 'opacity': 0.2,
    'xaxis': {
        # 'title': 'Time (\u03BCs)',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 1.5,
        'tickwidth': 1.5,
        'ticklen': 5,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white',
        },
    'yaxis': {
        # 'title': 'Coherence',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 1.5,
        'tickwidth': 1.5,
        'ticklen': 5,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white'
        },
    'font':{'family':'mathjax',
            'size': 30,
            }
}

In [4]:
def get_B1(t_pi):
    return (2*np.pi/28025)*(1/(2*t_pi))

###### References 

1. Zhiyang Yuan et. al. 'Charge state dynamics and optically detected electron spin resonance contrast of shallow nitrogen-vacancy centers in diamond", DOI: https://doi.org/10.1103/PhysRevResearch.2.033263
2. I. Panadero et. al. ""Photon-emission statistics for single nitrogen-vacancy centers, DOI: https://doi.org/10.1103/PhysRevApplied.22.014035"

In [5]:
def nv_parameters(gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, theta=np.pi/2, phi=0,Bx=0.0,By=0.0,Bz=0.0,gamma_d_in=0,gamma_d_out=0):
    Bx = Bx                          # Static magnetic field along x in tesla
    By = By                          # Static magnetic field along y in tesla
    Bz = Bz                          # Static magnetic field along z in tesla
    # c = 0.1                          # NV- Laser excitation constant in MHz/uW 
    c = 0.7                          # NV- Laser excitation constant in MHz/uW 
    
    gamma_p = c*power_laser          # NV- Laser excitation rate
    # gamma_0 = 63                   # NV- Radiative emission rate 
    gamma_0 = 10                     # NV- Radiative emission rate 
        
    gamma_s0_in = 12                 # NV- Excited 0 to singlet
    gamma_s1_in = 80                 # NV- Excited 1 to singlet
    # gamma_s1_in = 18               # NV- Excited 1 to singlet
    gamma_s0_out = 3.3               # NV- Singlet to ground 0 
    gamma_s1_out = 2.4               # NV- Singlet to ground 1

    if power_laser!=0:
        dark_ratio = 50/power_laser
    else:
        dark_ratio = 1

    # if gamma_d_in==0 or gamma_d_out==0:
    #     gamma_d_in = 0
    #     gamma_d_out = 0
        
    gamma_d0_in =  gamma_d_in
    gamma_d1_in =  gamma_d_in
    gamma_d0_out =  gamma_d_out
    gamma_d1_out = gamma_d_out

    # print(gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out)
    
    B1x = B1 * np.sin(theta) * np.cos(phi)
    B1y = B1 * np.sin(theta) * np.sin(phi)
    B1z = B1 * np.cos(theta)
    
    parameters = [Bx, By, Bz, gamma_rel, gamma_deph, gamma_p, gamma_0, gamma_s0_in, gamma_s1_in, gamma_s0_out, gamma_s1_out,
                  gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out, B1x, B1y, B1z]

    return parameters

In [6]:
def NV_spinNcharge(parameters, tlist, psi0, drive=-1,detuning_minus = 0.0,E=0.0):

    Bx, By, Bz, gamma_rel, gamma_deph, gamma_p, gamma_0, gamma_f0, gamma_f1, gamma_s0, gamma_s1, gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out, B1x, B1y, B1z = parameters

    # Hamiltonian Parameters
    D = 2870          # MHz
    g = 28025         # MHz/T
    E = E             # MHz

    # Operators
    n_states = 8
    spin_states = 3
    rho = np.zeros((n_states,n_states),dtype=complex)

    # Pauli Operators
    sx = rho.copy()
    sx[:spin_states,:spin_states] = jmat(1,'x').full()
    sx[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sx = Qobj(sx)
    sy = rho.copy()
    sy[:spin_states,:spin_states] = jmat(1,'y').full()
    sy[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sy = Qobj(sy)
    sz = rho.copy()
    sz[:spin_states,:spin_states] = jmat(1,'z').full()
    sz[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sz = Qobj(sz)
    s_plus = rho.copy()
    s_plus[:spin_states,:spin_states] = jmat(1,'+').full()
    s_plus[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    s_plus = Qobj(s_plus)
    s_minus = rho.copy()
    s_minus[:spin_states,:spin_states] = jmat(1,'-').full()
    s_minus[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    s_minus = Qobj(s_minus)

    # Other Operators
    # |g> --> |e>
    excitation = rho.copy()
    excitation[3,0] = np.sqrt(gamma_p)    # why this term is related to the specific transition? 
    excitation[4,1] = np.sqrt(gamma_p)
    excitation[5,2] = np.sqrt(gamma_p)
    excitation = Qobj(excitation)

    # |e> --> |g>
    emission = rho.copy()
    emission[0,3] = np.sqrt(gamma_0)
    emission[1,4] = np.sqrt(gamma_0)
    emission[2,5] = np.sqrt(gamma_0)
    emission = Qobj(emission)

    # |e+1> --> |s>
    singlet_in1p = rho.copy()
    singlet_in1p[6,3] = np.sqrt(gamma_f1) 
    singlet_in1p = Qobj(singlet_in1p)

    # |e0> --> |s>
    singlet_in0 = rho.copy()
    singlet_in0[6,4] = np.sqrt(gamma_f0) 
    singlet_in0 = Qobj(singlet_in0)

    # |e-1> --> |s>
    singlet_in1m = rho.copy()
    singlet_in1m[6,5] = np.sqrt(gamma_f1) 
    singlet_in1m = Qobj(singlet_in1m)

    # |s> --> |g+1>
    singlet_out1p = rho.copy()
    singlet_out1p[0,6] = np.sqrt(gamma_s1)  
    singlet_out1p = Qobj(singlet_out1p)

    # |s> --> |g0>
    singlet_out0 = rho.copy()
    singlet_out0[1,6] = np.sqrt(gamma_s0) 
    singlet_out0 = Qobj(singlet_out0)

    # |s> --> |g-1>
    singlet_out1m = rho.copy()
    singlet_out1m[2,6] = np.sqrt(gamma_s1) 
    singlet_out1m = Qobj(singlet_out1m)

    # |e+1> --> |d>
    dark_in1p = rho.copy()
    dark_in1p[7,3] = np.sqrt(gamma_d1_in) 
    dark_in1p = Qobj(dark_in1p)

    # |e0> --> |d>
    dark_in0 = rho.copy()
    dark_in0[7,4] = np.sqrt(gamma_d0_in) 
    dark_in0 = Qobj(dark_in0)

    # |e-1> --> |d>
    dark_in1m = rho.copy()
    dark_in1m[7,5] = np.sqrt(gamma_d1_in) 
    dark_in1m = Qobj(dark_in1m)

    # |d> --> |g+1>
    dark_out1p = rho.copy()
    dark_out1p[0,7] = np.sqrt(gamma_d1_out)  
    dark_out1p = Qobj(dark_out1p)

    # |d> --> |g0>
    dark_out0 = rho.copy()
    dark_out0[1,7] = np.sqrt(gamma_d0_out) 
    dark_out0 = Qobj(dark_out0)

    # |d> --> |g-1>
    dark_out1m = rho.copy()
    dark_out1m[2,7] = np.sqrt(gamma_d1_out) 
    dark_out1m = Qobj(dark_out1m)

    # Time-independent Hamiltonian
    H0 = D*(sz**2 - 2/3) + g*sz*Bz + g*(sx*Bx + sy*By) + E*(sx**2 - sy**2)

    # Solve for eigenenergies and eigenstates
    eigenenergies, eigenstates = H0.eigenstates()

    # print(eigenenergies)
    
    eigenenergies = np.unique(eigenenergies)
    # print(eigenenergies)

    if drive == -1:
        omega = eigenenergies[1]-eigenenergies[0]-detuning_minus
        # print(f"Resonant frequency is {omega} MHz")
    else:
        omega = eigenenergies[2]-eigenenergies[0]
        # print(f"Resonant frequency is {omega} MHz")
        

    H1 = g * (sx*B1x + sy*B1y + sz*B1z) 
    H_drive = [[H1, lambda t, args: np.cos(omega * t)]]

    # Total Hamiltonian
    H = [H0] + H_drive

    # Collapse operators
    c_ops = [
            np.sqrt(gamma_rel/(2*np.pi)) * s_minus,     # T1 Relaxation (Sigma_minus)   
            np.sqrt(gamma_rel/(2*np.pi)) * s_plus,      # T1 Relaxation (sigma_plus)    
            np.sqrt(2*gamma_deph) * sz,         # Dephasing                             
            excitation,                         # Excitation
            emission,                           # Emission
        
            singlet_in1p,                       # |e+1> --> |s>
            singlet_in0,                        # |e0> --> |s>
            singlet_in1m,                       # |e-1> --> |s>
            singlet_out1p,                      # |s> --> |g+1>
            singlet_out0,                       # |s> --> |g0>
            singlet_out1m,                      # |s> --> |g-1>
        
            dark_in1p,                          # |e+1> --> |d>
            dark_in0,                           # |e0> --> |d>
            dark_in1m,                          # |e-1> --> |d>
            dark_out1p,                         # |d> --> |g+1>
            dark_out0,                          # |d> --> |g0>
            dark_out1m,                         # |d> --> |g-1>
            ]


    # Observables to record
    expect_ops = [basis(n_states, 0)*basis(n_states, 0).dag(),  # |+1_g> Population
                basis(n_states, 1)*basis(n_states, 1).dag(),    # |0_g> Population
                basis(n_states, 2)*basis(n_states, 2).dag(),    # |-1_g> Population
                basis(n_states, 3)*basis(n_states, 3).dag(),    # |+1_e> Population
                basis(n_states, 4)*basis(n_states, 4).dag(),    # |0_e> Population
                basis(n_states, 5)*basis(n_states, 5).dag(),    # |-1_e> Population
                basis(n_states, 6)*basis(n_states, 6).dag(),    # |s> Population
                basis(n_states, 7)*basis(n_states, 7).dag(),    # |d> Population
                ]

    # Solve the master equation
    result = mesolve(H, psi0, tlist, c_ops, expect_ops,
                   options={"store_final_state": True, "nsteps": 1e5, "store_states": True},
                   )
    # result = mesolve(H, psi0, tlist, c_ops, expect_ops,
    #                options={"store_final_state": True, "nsteps": 1e5, "store_states": False},
    #                )

    return result.expect, result.final_state, result.states
    # return result.expect, _, result.final_state
    
    # return eigenenergies  

In [7]:
# def delay(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
#                gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0, duration=5, drive=-1, t_steps=1001, gamma_d_in=0,gamma_d_out=0):
    
#     parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    
#     # Time array
#     tlist = np.linspace(0, duration, t_steps)  # Time in us
    
#     populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
#     return populations, psi_list, tlist   

# ########################################################################################################

def initialize(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, duration=5, drive=-1, t_steps=1001, gamma_d_in=0,gamma_d_out=0):
    
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
    return populations, psi_list, tlist   

########################################################################################################

def rabi(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0.00164, duration=5, 
         drive=-1, theta=np.pi/2, phi=0, t_steps=101,detuning_minus=0.0, gamma_d_in=0,gamma_d_out=0):
    # print('value of B1 in function rabi: ',B1)
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1, theta, phi,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive,detuning_minus)
    
    return populations, psi_list, tlist  

########################################################################################################

def evolve(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0, duration=0.1, drive=-1, t_steps=101,spacing_type = 'linspace', gamma_d_in=0,gamma_d_out=0,t_start = 0.02):

    # print('value of B1 in function evolve: ',B1)
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    if spacing_type == 'geomspace':
        tlist = np.geomspace(t_start,duration,t_steps)
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
    return populations, psi_list, tlist  

########################################################################################################

def read(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, duration=5, drive=-1, t_steps=1001, gamma_d_in=0,gamma_d_out=0):
    
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    psi0 = Qobj(qdiags(psi0.diag(), offsets=0), dims=psi0.dims)        # removing the off-diagonal terms as they are not relevant                
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
    return populations, psi_list, tlist  

########################################################################################################

def count(populations,tlist,duration=0.5, gamma_0=63, collection=0.01):
    index = np.argwhere(tlist<=duration)[-1][0]
    # print(index)
    n = np.mean(np.array(np.real(populations[3:6])[:,:index+1]))
    photons = n*duration*gamma_0*collection
    return photons

Initial state

In [8]:
# psi0 = (1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2))

#### Actual $T_1$

In [9]:
# # power_array = np.array([5,15,24])
# # init_array = np.array([15,50,100])
# power_array = np.array([8])
# init_array = np.array([25])

# gamma_d_in = np.linspace(8,500,20)
# gamma_d_out = np.linspace(0.3,30,20)
# # gamma_d_in = np.array([0])
# # gamma_d_out = np.array([0])

# dark_steps = 51

In [10]:
# # Parameters
# gamma_rel = 0.02      # keeping T1 lifetime = 50 us
# gamma_deph = 0.0     
# read_window = 0.3
# dark_time = 100

# signal_array = np.zeros((power_array.shape[0],init_array.shape[0], gamma_d_in.shape[0],gamma_d_out.shape[0],dark_steps))
# t_array = np.zeros((power_array.shape[0],init_array.shape[0], gamma_d_in.shape[0],gamma_d_out.shape[0],dark_steps))


# for p, laser_power in tqdm(enumerate(power_array), total=len(power_array), position=0, leave=True, desc="power_array"):
#     for t, t_init in tqdm(enumerate(init_array), total=len(init_array), position=0, leave=True, desc="init_array"):
#         for d1, d_in in tqdm(enumerate(gamma_d_in), total=len(gamma_d_in), position=0, leave=True, desc="gamma_d_in"):
#             for d2, d_out in tqdm(enumerate(gamma_d_out), total=len(gamma_d_out), position=0, leave=True, desc="gamma_d_out"):

                
#                 _,psi1,_=initialize(psi0,power_laser=laser_power,duration=t_init,t_steps=51,
#                                     gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                 populations1,psi2,t_t1=evolve(psi1[-1], duration=dark_time,t_steps=dark_steps,
#                                               spacing_type = 'geomspace', gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                 t_array[p,t,d1,d2,:] = t_t1
                
#                 for i in range(len(psi2)):
#                     populations3,psi3,t_read=read(psi2[i],power_laser=laser_power,duration=read_window,t_steps=51,
#                                                   gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                     photons = count(populations3, t_read, duration=read_window) 
#                     signal_array[p,t,d1,d2,i] = photons

In [20]:
from joblib import Parallel, delayed

def compute_signal(args):
    try:
        params, psi0, power_array, init_array, gamma_d_in, gamma_d_out, \
        gamma_rel, gamma_deph, read_window, dark_time, dark_steps = args

        p, t, d1, d2 = params
        laser_power = power_array[p]
        t_init = init_array[t]
        d_in = gamma_d_in[d1]
        d_out = gamma_d_out[d2]

        _, psi1, _ = initialize(psi0, power_laser=laser_power, duration=t_init, t_steps=51,
                                gamma_rel=gamma_rel, gamma_deph=gamma_deph,
                                gamma_d_in=d_in, gamma_d_out=d_out)

        populations1, psi2, t_t1 = evolve(psi1[-1], duration=dark_time, t_steps=dark_steps,
                                          spacing_type='geomspace', t_start=0.02, gamma_rel=gamma_rel, gamma_deph=gamma_deph,
                                          gamma_d_in=d_in, gamma_d_out=d_out)

        signal_local = np.zeros(dark_steps)

        for i in range(len(psi2)):
            populations3, psi3, t_read = read(psi2[i], power_laser=laser_power, duration=read_window, t_steps=51,
                                              gamma_rel=gamma_rel, gamma_deph=gamma_deph,
                                              gamma_d_in=d_in, gamma_d_out=d_out)
            photons = count(populations3, t_read, duration=read_window)
            signal_local[i] = photons

        return (p, t, d1, d2, t_t1, signal_local)

    except Exception as e:
        import traceback
        print(f"Error in params {params}: {e}")
        traceback.print_exc()
        raise

if __name__ == "__main__":

    psi0 = (1/6)*ket2dm(basis(8, 0)) + (1/6)*ket2dm(basis(8, 1)) + (1/6)*ket2dm(basis(8, 2)) + (1/2)*ket2dm(basis(8, 7))

    power_array = np.array([5])
    init_array = np.array([15])
    # gamma_d_in = np.array([0,50])
    # gamma_d_out = np.linspace(0.02,1,100)
    gamma_d_in = np.array([0])
    gamma_d_out = np.array([0])
    dark_steps = 51

    gamma_rel = 0.02
    gamma_deph = 0.0
    read_window = 0.5
    dark_time = 100

    # Allocate output arrays
    signal_array = np.zeros((power_array.shape[0], init_array.shape[0], gamma_d_in.shape[0], gamma_d_out.shape[0], dark_steps))
    t_array = np.zeros_like(signal_array)

    # Prepare list of all parameters to pass to worker function
    params_list = [(p, t, d1, d2)
                   for p in range(len(power_array))
                   for t in range(len(init_array))
                   for d1 in range(len(gamma_d_in))
                   for d2 in range(len(gamma_d_out))]

    # Pack all arguments into tuples because Pool.map expects one argument per call
    args_list = [(params, psi0, power_array, init_array, gamma_d_in, gamma_d_out,
                  gamma_rel, gamma_deph, read_window, dark_time, dark_steps) for params in params_list]

    results = Parallel(n_jobs=-1)(
        delayed(compute_signal)(arg) for arg in tqdm(args_list)
    )
    
    for p, t, d1, d2, t_t1, signal_local in results:
        t_array[p, t, d1, d2, :] = t_t1
        signal_array[p, t, d1, d2, :] = signal_local
    
    print("Completed processing.")

100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 250.48it/s]


Completed processing.


In [21]:
folder_path = 'data'
os.makedirs(folder_path, exist_ok=True)
# filename =  os.path.join(folder_path, f'{timestamp}_5uW_15us_readout_0.5us_gamma_d_in_0_dark_halfPop.npz')
filename =  os.path.join(folder_path, f'5uW_15us_readout_0.5us_gamma_d_in_0_dark_halfPop.npz')

np.savez_compressed(filename, signal_array=signal_array, t_array=t_array)

In [22]:
# filename = r"D:\Atanu\NV_laser_dynamics\NV_LaserDynamics\NV_simulator\probing_metastable\data\48-56_5uW_15us_readout_0.1us.npz"

sim_data = np.load(filename)
sim_signal_array = sim_data['signal_array']
sim_t_array = sim_data['t_array']

sim_signal_array = np.squeeze(sim_signal_array)
sim_t_array = np.squeeze(sim_t_array)*1e3        #in ns

sim_signal_array.shape, sim_t_array.shape

((51,), (51,))

In [23]:
exp_filename = r"D:\Atanu\Atanu_Github\T1_measurements\codes\data\2025\May_12\arranging_datas_with_power\5uW\l_15.0us\[17_22_10]_1.0us.npz"
exp_data = np.load(exp_filename)
time_axis, signal_photon, _, _ = data_to_time_signal(exp_data, 2000, first='signal')

time_axis = time_axis[1:]
signal_photon = signal_photon[1:]

signal_photon.shape,time_axis.shape

((49,), (49,))

In [24]:
std_t_array = np.array([0.02,0.05,0.1,0.2,0.51,2.5,5,10,20,50,100])*1e3     # in ns    

In [25]:
# Interpolate experimental data once
exp_interp = interp1d(time_axis, signal_photon, bounds_error=False, fill_value="extrapolate")
exp_values_at_std = exp_interp(std_t_array)

# Initialize MAE result array
mae_array = np.zeros((gamma_d_in.shape[0], gamma_d_out.shape[0]))

# Compute MAE for each simulation
for i in range(gamma_d_in.shape[0]):
    for j in range(gamma_d_out.shape[0]):
        # Extract and normalize simulation data
        sim_time = sim_t_array[i, j, :]
        sim_signal = sim_signal_array[i, j, :]
        sim_signal_normalized = sim_signal / sim_signal[0] if sim_signal[0] != 0 else sim_signal

        # Interpolate simulation signal
        sim_interp = interp1d(sim_time, sim_signal_normalized, bounds_error=False, fill_value="extrapolate")
        sim_values_at_std = sim_interp(std_t_array)

        # Compute MAE
        mae_array[i, j] = np.mean(np.abs(sim_values_at_std - exp_values_at_std))

print("MAE array shape:", mae_array.shape)

# Find lowest 5 MAEs for each gamma_d_in[i]
top5_indices_per_row = np.argsort(mae_array, axis=1)[:, :5]
top5_maes_per_row = np.take_along_axis(mae_array, top5_indices_per_row, axis=1)

# Display per-row top 5 results
for i in range(mae_array.shape[0]):
    print(f"\ngamma_d_in[{i}] = {gamma_d_in[i]:.4g}:")
    for rank in range(5):
        j = top5_indices_per_row[i, rank]
        mae_val = top5_maes_per_row[i, rank]
        print(f"  Rank {rank+1}: gamma_d_out[{j}] = {gamma_d_out[j]:.4g}, MAE = {mae_val:.6f}")

# ====== New Section: Find Global Minimum MAE ======
min_index = np.unravel_index(np.argmin(mae_array), mae_array.shape)
min_i, min_j = min_index
min_mae = mae_array[min_i, min_j]

print("\n==============================")
print("Global Minimum MAE Details:")
print(f"gamma_d_in[{min_i}] = {gamma_d_in[min_i]:.4g}")
print(f"gamma_d_out[{min_j}] = {gamma_d_out[min_j]:.4g}")
print(f"Minimum MAE = {min_mae:.6f}")
print("==============================")

IndexError: too many indices for array: array is 1-dimensional, but 3 were indexed

In [26]:
min_i = 0
min_j = 1

# === Get the simulation data for the global minimum MAE ===
best_sim_time = sim_t_array[min_i, min_j, :]
best_sim_signal = sim_signal_array[min_i, min_j, :]
best_sim_signal_norm = best_sim_signal / best_sim_signal[0] if best_sim_signal[0] != 0 else best_sim_signal

# === Normalize experimental data ===
exp_signal_norm = signal_photon / signal_photon[0] if signal_photon[0] != 0 else signal_photon

# === Create Plotly figure ===
fig = go.Figure()

# Plot simulation curve (best fit)
fig.add_scatter(
    x=best_sim_time / 1000,  # Convert ns to μs
    y=best_sim_signal_norm,
    mode='lines+markers',
    # name=f'Best Sim: γ_in={gamma_d_in[min_i]:.2f} MHz, γ_out={gamma_d_out[min_j]:.2f} MHz',
    line=dict(width=3)
)

# Plot experimental curve
fig.add_scatter(
    x=time_axis / 1000,  # Convert ns to μs
    y=exp_signal_norm,
    mode='lines+markers',
    name='Experimental',
    marker=dict(symbol='circle', size=6),
    line=dict(width=3, dash='dash')
)

fig.update_layout(
    template=fig_template,
    height=800,
    width=1600,
    xaxis=dict(title="Time (μs)", range=[0, 100]),  # Limit x-axis to 0–100 μs
    yaxis=dict(title="Normalized Photon Counts (arb. unit)"),
    legend=dict(itemsizing='constant', font=dict(size=10), bgcolor='rgba(255,255,255,0.8)', bordercolor="LightGray", borderwidth=1)
)

fig.update_yaxes(automargin=True)
# fig.update_xaxes(type="log")

# Show the plot
fig.show()

IndexError: too many indices for array: array is 1-dimensional, but 3 were indexed

In [141]:
1/0.065

15.384615384615383

In [20]:
# fig = go.Figure()
# for p, laser_power in enumerate(power_array):
#     for t, t_init in enumerate(init_array):
#         for d1, d_in in enumerate(gamma_d_in):
#             for d2, d_out in enumerate(gamma_d_out):
#                 # fig.add_scatter(x=sim_t_array[d1,d2,:], y=sim_signal_array[d1,d2,:],name=f'Sig_pow_{laser_power}uW_init_{t_init}us_d_in_{np.round(d_in,2)}MHz_d_out_{np.round(d_out,2)}MHz',mode='lines+markers',line=dict(width=3))
#                 fig.add_scatter(x=sim_t_array[d1,d2,:], y=sim_signal_array[d1,d2,:]/sim_signal_array[d1,d2,0],name=f'NormSig_pow_{laser_power}uW_init_{t_init}us_din_{np.round(d_in,2)}MHz_dout_{np.round(d_out,2)}MHz',mode='lines+markers',line=dict(width=3))

# fig.update_layout(template=fig_template,
#                   height=700, width=1500,
#                   # legend=dict(yanchor="top",y=0.98,xanchor="left",x=1.01),
#                   xaxis=dict(title="Time(\N{greek small letter mu}s)"),
#                   yaxis=dict(title="Photon Counts (arb. unit)"),
#                   title="NV<sup>-1</sup> T<sub>1</sub> Signal",
#                   )
# fig.update_yaxes(automargin=True)
# # fig.write_html(folder_path + "all_middle_values_readout_0.3us.html")
# fig

cross check 

In [ ]:
# # Parameters
# gamma_rel = 0.02   # keeping T1 lifetime = 50 us
# gamma_deph = 0.0    
# t_init = 100     
# laser_power = 15  
# read_window = 0.5
# dark_time = 100

# _,psi1,_=initialize(psi0,power_laser=laser_power,duration=t_init,t_steps=51,gamma_rel=gamma_rel,gamma_deph=gamma_deph)
# populations1,psi2,t_t1=evolve(psi1[-1], duration=dark_time,t_steps=51,spacing_type = 'geomspace', gamma_rel=gamma_rel,gamma_deph=gamma_deph)

# signal_t1 = []
# pop_read_t1 = []

# for i in tqdm(range(len(psi2))):
#     populations3,psi3,t_read=read(psi2[i],power_laser=laser_power,duration=read_window,t_steps=51,gamma_rel=gamma_rel,gamma_deph=gamma_deph)
#     photons = count(populations3, t_read, duration=read_window)
#     signal_t1.append(photons)
#     pop_read_t1.append(populations3)

# signal_t1 = np.array(signal_t1)
# pop_read_t1 = np.array(pop_read_t1)